Command() is a control primitive.

It is a LangGraph control primitive that allows a node to update state and dynamically control where the graph should execute next.

A quick summary:

Workflow= overall execution structure

Agent = decision-making component

Command = mechanism to control graph execution

Command() allows a node to update the state + choose the next node + sometimes resume an interrupted workflow in a single return.

In a normal workflow, routing is defined in the graph definition:

```
Node A
  ↓
Conditional Edge
  ↓
Node B / Node C
```

In [ ]:
def router(state):
    return state["route"]

builder.add_conditional_edges(
    "router",
    router,
    {
        "billing": "billing",
        "technical": "technical"
    }
)

Here, the node only updates the state, and the conditional edge handles routing separately.

With Command(), the same node can specify:

In [ ]:
return Command(
    goto="billing",
    update={
        "active_agent": "billing"
    }
)

Current Node
    ↓
Command()
    ├── update state
    └── goto next node

This means routing and state update happen in the same place.

Difference from a workflow
Normal predefined workflow:

START
 ↓
A
 ↓
B
 ↓
C
 ↓
END

The developer defines the path while building the graph.

Command():
Node A
  ↓
runtime decision
  ↓
Command(goto="B")

or

Command(goto="C")

So Command() makes dynamic control flow easier.

Suppose the LLM decides:
User request
   ↓
LLM
   ↓
"Billing agent should handle this"
   ↓
Command(goto="billing_agent")

Here:
LLM= decision maker

Command() = mechanism for applying the decision to graph execution

This distinction is important.
Command() is not an agent.
The agent makes the decision; Command() controls the graph based on that decision.

Especially useful in a handoff

A handoff means transferring control from one agent to another specialized agent so that the new agent continues handling the task.

In [ ]:
return Command(
    goto=decision.target,
    update={
        "active_agent": decision.target
    }
)

means:
Triage Agent
   ↓
LLM decides:
technical_agent
   ↓
Command()
   ├── active_agent = technical_agent
   └── goto technical_agent
   ↓
Technical Agent

Roughly the same thing with a conditional edge:
Triage
 ↓
update state
 ↓
routing function
 ↓
conditional edge
 ↓
Technical Agent

Command():
Triage
 ↓
Command(update + goto)
 ↓
Technical Agent

Command vs Send
This is also important:

Command
→ move control to the next node
→ usually one routing/action
→ can also update the state

Send
→ dynamically create multiple executions
→ send custom input to each worker


In [ ]:
# Example:
# Command(
#     goto="technical_agent"
# )  
# means:
# Go to technical_agent

# Whereas:
# [
#     Send("worker", {"topic": "RAG"}),
#     Send("worker", {"topic": "Agents"}),
#     Send("worker", {"topic": "Memory"})
# ]

# means:
# Run worker three times
# with three different inputs

One more use: HITL resume
An important use of Command():

Command(
    resume="approved"
)

It is also used to resume an interrupted graph.
So mentally:
Command()
   ├── goto     → where next?
   ├── update   → what state changes?
   └── resume   → resume interrupted workflow